In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.ticker import LogLocator
import matplotlib.colors as mcolors

import os

# Speed Score Comparison - Configuration Guide

This notebook allows flexible comparison of models across different Akida versions.

## Configuration Options

### 1. Akida Versions
Enable/disable different Akida versions by setting `"enabled": True/False` in the `AKIDA_VERSIONS` dict:
- **Compare both V1 and V2**: Set both to `True`
- **Only V1**: Set v1 to `True`, v2 to `False`
- **Only V2**: Set v1 to `False`, v2 to `True`

### 2. Model Types
Select which model types to include in `ENABLED_MODEL_TYPES`:
- `"float"` - Floating point models
- `"qat"` - Quantization-aware trained models
- `"deployed"` - Deployed/quantized models

**Examples:**
```python
ENABLED_MODEL_TYPES = ["float", "qat", "deployed"]  # All types
ENABLED_MODEL_TYPES = ["qat", "deployed"]           # Only quantized models
ENABLED_MODEL_TYPES = ["float"]                     # Only float models
```

### 3. Architectures
Select which architectures to include in `ENABLED_ARCHITECTURES`:
- `"lnes"` - LNES representation
- `"two_d_histogram"` - 2D histogram representation
- `"event_frame"` - Event frame representation

**Examples:**
```python
ENABLED_ARCHITECTURES = ["lnes", "two_d_histogram", "event_frame"]  # All
ENABLED_ARCHITECTURES = ["lnes"]                                    # Only LNES
```

### 4. Plot Customization
- **PLOT_TITLE**: Change the plot title
- **PLOT_OFFSET**: Number of samples to exclude from tail (-10 = exclude last 10)

## Visual Legend
- **Colors**: Different model types (float=blue, qat=red, deployed=green)
- **Shades**: Different Akida versions (V1=darker, V2=lighter)
- **Line styles**: Different architectures (solid, dashed, dash-dot)


## Quick Configuration Examples

Uncomment and run one of these examples to quickly set up common comparisons:

### Example 1: Compare V1 vs V2 (all models)
```python
# Already configured by default
```

### Example 2: Only Akida V2 models
```python
# AKIDA_VERSIONS["v1"]["enabled"] = False
# AKIDA_VERSIONS["v2"]["enabled"] = True
```

### Example 3: Only deployed models across V1 and V2
```python
# ENABLED_MODEL_TYPES = ["deployed"]
# PLOT_TITLE = "Akida V1 vs V2 - Deployed Models Only"
```

### Example 4: Compare float vs deployed (V2 only)
```python
# AKIDA_VERSIONS["v1"]["enabled"] = False
# ENABLED_MODEL_TYPES = ["float", "deployed"]
# PLOT_TITLE = "Akida V2 - Float vs Deployed"
```

### Example 5: Only LNES architecture across versions
```python
# ENABLED_ARCHITECTURES = ["lnes"]
# PLOT_TITLE = "Akida V1 vs V2 - LNES Architecture"
```


In [ ]:
# ============================
# CONFIGURATION
# ============================

# Akida versions to include (can select multiple)
AKIDA_VERSIONS = {
    "v1": {
        "enabled": True,
        "base_path": "~/Data/astrospikes/Astrospike/10_27_2025/share_astro_results/v1",
        "label_prefix": "V1"
    },
    "v2": {
        "enabled": True,
        "base_path": "~/Data/astrospikes/Astrospike/10_27_2025/share_astro_results/v2",
        "label_prefix": "V2"
    }
}

# Model types to include in comparison
# Options: 'float', 'qat', 'deployed'
ENABLED_MODEL_TYPES = ["qat"]

# Architectures to include in comparison
# Options: 'lnes', 'two_d_histogram', 'event_frame'
ENABLED_ARCHITECTURES = ["lnes", "two_d_histogram", "event_frame"]

# Plot title
PLOT_TITLE = "Akida V1 vs V2 - Quantized Models"

# Plot settings
PLOT_OFFSET = -10  # Number of samples to exclude from the tail


In [ ]:
def load_csv_files_multi_version(akida_versions, enabled_model_types, enabled_architectures):
    dict_files = {}
    
    for version_key, version_config in akida_versions.items():
        if not version_config.get("enabled", False):
            continue
            
        base_path = os.path.expanduser(version_config["base_path"])
        
        if not os.path.exists(base_path):
            print(f"Warning: Path does not exist for {version_key}: {base_path}")
            continue
        
        for root, _, files in os.walk(base_path):
            for file in files:
                if file.endswith('_metrics.csv'):
                    csv_path = os.path.join(root, file)
                    rel_path = os.path.relpath(csv_path, base_path)
                    
                    # Parse and filter based on configuration
                    model_type, architecture = parse_model_info(rel_path)
                    
                    if model_type in enabled_model_types and architecture in enabled_architectures:
                        df = pd.read_csv(csv_path)
                        # Use tuple of (version, rel_path) as key
                        dict_files[(version_key, rel_path)] = df
    
    return dict_files

# This function needs to be defined before use, so moving it here
def parse_model_info(path):
    parts = path.split("/")
    model_type = parts[0] if len(parts) > 0 else "unknown"
    
    # Extract architecture from path
    if "lnes" in path.lower():
        architecture = "lnes"
    elif "two_d_histogram" in path:
        architecture = "two_d_histogram"
    elif "event_frame" in path:
        architecture = "event_frame"
    else:
        architecture = "unknown"
    
    return model_type, architecture

# Load CSV files with configuration
dict_files = load_csv_files_multi_version(AKIDA_VERSIONS, ENABLED_MODEL_TYPES, ENABLED_ARCHITECTURES)
print(f"Loaded {len(dict_files)} CSV files.")

# Print summary by version
for version_key in AKIDA_VERSIONS.keys():
    if AKIDA_VERSIONS[version_key].get("enabled", False):
        count = sum(1 for k in dict_files.keys() if k[0] == version_key)
        print(f"  - {AKIDA_VERSIONS[version_key]['label_prefix']}: {count} files")

In [ ]:
def plot_speed_score_cdf(df, ax, label, color, linestyle="-", offset=-5, linewidth=2.0, alpha=1.0):
    # Extract and sort speed scores
    scores = df["speed_score"].to_numpy()
    scores.sort()
    n = len(scores)
    
    # Compute cumulative distribution
    y = np.arange(1, n + 1) / n

    x_plot, y_plot = scores[:offset], y[:offset]

    # Plot on the given axes with specified style
    ax.plot(x_plot, y_plot, linestyle=linestyle, linewidth=linewidth, 
            label=label, color=color, alpha=alpha)


In [ ]:
def get_variant(rel_path):
    if "results_5" in rel_path:
        return "5"
    elif "results_8" in rel_path:
        return "8"
    return None

def get_line_style(architecture):
    styles = {
        "lnes": "-",              # Solid line
        "two_d_histogram": "--",  # Dashed line
        "event_frame": "-.",      # Dash-dot line
        "unknown": ":"            # Dotted line
    }
    return styles.get(architecture, "-")

def create_short_label(version_key, rel_path, model_type, architecture, akida_versions):
    version_prefix = akida_versions[version_key]["label_prefix"]
    label = f"{version_prefix} - {model_type.upper()} - {architecture.replace('_', ' ').title()}"
    
    # Add results identifier if present (results_5, results_8, etc.)
    if "results_5" in rel_path:
        label += " (5)"
    elif "results_8" in rel_path:
        label += " (8)"
    
    return label

def get_color_for_model(version_key, model_type, architecture, num_versions):
    # V1 colors (darker, cooler tones)
    v1_colors = {
        ("float", "lnes"): [0.12, 0.47, 0.71],           # Blue
        ("float", "two_d_histogram"): [0.0, 0.2, 0.4],   # Dark blue
        ("float", "event_frame"): [0.4, 0.76, 1.0],      # Light blue
        ("qat", "lnes"): [0.89, 0.10, 0.11],             # Red
        ("qat", "two_d_histogram"): [0.6, 0.0, 0.0],     # Dark red
        ("qat", "event_frame"): [1.0, 0.5, 0.5],         # Light red
        ("deployed", "lnes"): [0.20, 0.63, 0.17],        # Green
        ("deployed", "two_d_histogram"): [0.0, 0.4, 0.0],# Dark green
        ("deployed", "event_frame"): [0.5, 1.0, 0.5],    # Light green
    }
    
    # V2 colors (distinct warm tones)
    v2_colors = {
        ("float", "lnes"): [1.0, 0.5, 0.0],              # Orange
        ("float", "two_d_histogram"): [0.8, 0.3, 0.0],   # Dark orange
        ("float", "event_frame"): [1.0, 0.7, 0.3],       # Light orange
        ("qat", "lnes"): [0.8, 0.2, 0.8],                # Magenta
        ("qat", "two_d_histogram"): [0.5, 0.0, 0.5],     # Dark magenta
        ("qat", "event_frame"): [1.0, 0.5, 1.0],         # Light magenta
        ("deployed", "lnes"): [0.6, 0.4, 0.8],           # Purple
        ("deployed", "two_d_histogram"): [0.4, 0.2, 0.6],# Dark purple
        ("deployed", "event_frame"): [0.8, 0.6, 1.0],    # Light purple
    }
    
    color_map = v1_colors if version_key == "v1" else v2_colors
    color = color_map.get((model_type, architecture), [0.5, 0.5, 0.5])
    
    return color

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
ax.set_title(PLOT_TITLE, fontsize=14)

def get_model_type_order(model_type):
    order_map = {"deployed": 0, "qat": 1, "float": 2, "unknown": 3}
    return order_map.get(model_type, 3)

def get_version_order(version_key):
    order_map = {"v1": 0, "v2": 1}
    return order_map.get(version_key, 99)

# Count enabled versions for color calculation
num_enabled_versions = sum(1 for v in AKIDA_VERSIONS.values() if v.get("enabled", False))

# Sort items: first by version, then by model type
sorted_items = sorted(dict_files.items(), 
                     key=lambda x: (get_version_order(x[0][0]), 
                                   get_model_type_order(parse_model_info(x[0][1])[0])))

# Plot each model
for (version_key, rel_path), df in sorted_items:
    model_type, architecture = parse_model_info(rel_path)
    label = create_short_label(version_key, rel_path, model_type, architecture, AKIDA_VERSIONS)
    color = get_color_for_model(version_key, model_type, architecture, num_enabled_versions)
    linestyle = get_line_style(architecture)
    
    # Differentiate variants by linewidth and alpha
    variant = get_variant(rel_path)
    linewidth = 2.5 if variant == "5" else 2.0
    alpha = 0.7 if variant == "5" else 1.0
    
    plot_speed_score_cdf(df, ax, label=label, color=color, 
                         linestyle=linestyle, offset=PLOT_OFFSET,
                         linewidth=linewidth, alpha=alpha)

# Configure axes
ax.set_xlabel("SPEED Score", fontsize=12)
ax.set_ylabel("Fraction of Test Set ($\leq s$)", fontsize=12)
ax.set_yscale('logit')
ax.set_ylim(0.001, 0.999)

# Set y-axis ticks
ticks = [0.001, 0.005, 0.01, 0.02, 0.05, 0.1, 0.5, 0.8, 0.9, 0.95, 0.98, 0.99, 0.995, 0.999]
ax.yaxis.set_major_locator(mticker.FixedLocator(ticks))
ax.yaxis.set_major_formatter(mticker.FixedFormatter([f'{t:g}' for t in ticks]))

# Styling
ax.grid(True, which='both', alpha=0.3)
ax.tick_params(axis='both', which='major', labelsize=10)
ax.legend(loc='lower right', fontsize=9)

# Save and display
plt.tight_layout()
output_dir = os.path.join(".", "speedscore_plots")
os.makedirs(output_dir, exist_ok=True)
plt.savefig(os.path.join(output_dir, PLOT_TITLE + ".png"), dpi=150, bbox_inches='tight')
plt.show()